# Can we predict next-month arrears?
Target: `arrears_next_month`. Follows `checklist.md`.

**Feature decisions (checklist 7):** `meter_ref` is an identifier (used only for grouping). `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time and is almost certainly a consequence of arrears: dropped. Remaining features (tariff, occupants, kwh, bill_eur, late_payments_12m) are recorded up to the billing month.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.base import clone
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
print(df.shape, df.arrears_next_month.mean())
print(df.groupby('billing_month').arrears_next_month.mean().round(3).iloc[::4])

## 1. Test set (set aside before any modelling)
The model will be used on later months, and accounts repeat across rows. So the test set is the **latest 5 months, restricted to a random 20% of accounts held out entirely**. Training rows are the other accounts' rows before the cutoff. Held-out accounts' earlier rows and other accounts' later rows are discarded, so no account and no future period leaks into training.

In [ ]:
rng=np.random.RandomState(0)
ents=df.meter_ref.unique(); test_ents=set(rng.choice(ents,int(.2*len(ents)),replace=False))
cutoff=sorted(df.billing_month.unique())[-5]
is_test_ent=df.meter_ref.isin(test_ents)
test=df[is_test_ent & (df.billing_month>=cutoff)].copy()
train=df[~is_test_ent & (df.billing_month<cutoff)].copy()
print(cutoff, len(train), len(test), train.arrears_next_month.mean(), test.arrears_next_month.mean())
assert not set(train.meter_ref)&set(test.meter_ref)
FEATS=['tariff','occupants','kwh','bill_eur','late_payments_12m']; TARGET='arrears_next_month'
Xtr,ytr=train[FEATS],train[TARGET]

## 4. Candidates (named before any scoring)
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: standardised numerics + one-hot tariff, logistic regression, C in {0.1, 1, 10}.
3. `hgb`: gradient boosting, max_depth in {2, 4}, learning_rate 0.05.

Metric for selection: PR-AUC (average precision), since arrears is ~12% prevalence; ROC-AUC reported too. The decision threshold is chosen to maximise F1 on out-of-fold predictions.

In [ ]:
pre=lambda: ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['tariff']),
    ('n',StandardScaler(),['occupants','kwh','bill_eur','late_payments_12m'])])
cands={'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C in [.1,1,10]: cands[f'logreg_C{C}']=Pipeline([('p',pre()),('m',LogisticRegression(C=C,max_iter=1000))])
for d in [2,4]: cands[f'hgb_d{d}']=Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=.05,max_iter=200,random_state=0))])
list(cands)

## 2-3. Cross-validation inside training rows
Same structure as the test split: each fold validates on a later block of months using a held-out subset of accounts; it trains only on earlier months of the other accounts. Preprocessing lives inside the pipelines, so it is fit per fold.

In [ ]:
months=sorted(train.billing_month.unique())
tr_ents=train.meter_ref.unique(); gid=dict(zip(tr_ents, np.random.RandomState(1).randint(0,4,len(tr_ents))))
g=train.meter_ref.map(gid).values
def folds():
    k=4; blocks=np.array_split(months,k+1)   # expanding window over time
    for i in range(1,k+1):
        vm=blocks[i]; tm=np.concatenate(blocks[:i])
        va=(train.billing_month.isin(vm)&(g==i-1)).values
        tr=(train.billing_month.isin(tm)&(g!=i-1)).values
        yield np.where(tr)[0],np.where(va)[0]
FOLDS=list(folds()); print([(len(a),len(b)) for a,b in FOLDS])
oof={}; res=[]
for name,m in cands.items():
    p=np.full(len(train),np.nan); sc=[]
    for a,b in FOLDS:
        mm=clone(m).fit(Xtr.iloc[a],ytr.iloc[a]); p[b]=mm.predict_proba(Xtr.iloc[b])[:,1]
        sc.append((average_precision_score(ytr.iloc[b],p[b]),roc_auc_score(ytr.iloc[b],p[b])))
    oof[name]=p; s=np.array(sc); res.append((name,*s.mean(0),*s.std(0)))
cv=pd.DataFrame(res,columns=['model','PR_AUC','ROC_AUC','PR_sd','ROC_sd']).set_index('model'); cv.round(4)

## 5. Choice (CV only)

In [ ]:
best=cv.PR_AUC.drop('baseline').idxmax()
mask=~np.isnan(oof[best]); ths=np.linspace(.05,.8,76)
f1s=[f1_score(ytr[mask],oof[best][mask]>t) for t in ths]; thr=float(ths[int(np.argmax(f1s))])
print('chosen:',best,'threshold:',round(thr,2),'CV F1:',round(max(f1s),3))

**Recorded choice:** the model above (best mean CV PR-AUC among non-baseline candidates), features = tariff, occupants, kwh, bill_eur, late_payments_12m, and the F1-optimal out-of-fold threshold printed above. Nothing below changes these.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=clone(cands[best]).fit(Xtr,ytr)
pt=final.predict_proba(test[FEATS])[:,1]; yt=test[TARGET]
out=pd.DataFrame({'chosen':[average_precision_score(yt,pt),roc_auc_score(yt,pt),f1_score(yt,pt>thr)],
  'baseline':[yt.mean(),.5,f1_score(yt,np.zeros(len(yt)))]},index=['PR_AUC','ROC_AUC',f'F1@{thr:.2f}']).round(4)
print('test rows',len(test),'prevalence',round(yt.mean(),3)); out

## Conclusion
See the table above (headline = test score of the chosen model against the prevalence baseline). Caveats: `final_notice_sent` was deliberately excluded because it is not available at prediction time; using it would inflate performance. The test set is a single split of ~275 accounts over 5 months, so the headline carries sampling uncertainty.